In [ ]:
# In-tree BFCL evaluator (Phase 4 redo). Skips the canonical bfcl-eval CLI
# entirely -- that path is deferred pending vllm/transformers/Qwen 3.5/NCCL
# stabilisation on Colab (see project memory + ADR 0005). This notebook
# runs through Unsloth in the same `[colab]` env we used for Phase 3
# training, evaluates both base and adapter on 11 BFCL V3 single-turn
# categories with an in-tree AST-match + decision-to-call scorer, and
# pushes the regenerated model card + chart to HF when the gate passes.
!git clone https://github.com/sukhrobnurali/tooltuned-qwen.git
%cd tooltuned-qwen
!pip install -e ".[colab]" --quiet
!pip uninstall -y torchcodec --quiet

In [ ]:
import sys, os
# Phase 1.3 finding #9: editable install + kernel restart can drop our
# package from sys.path -- belt and braces.
sys.path.insert(0, "/content/tooltuned-qwen/src")
# Colab secrets land in `userdata`, not os.environ (Phase 2 finding #10).
from google.colab import userdata
for key in ("HF_TOKEN",):
    val = userdata.get(key)
    assert val, f"Set {key} in Colab secrets and toggle Notebook access on"
    os.environ[key] = val
# Per-category truncation. 50 keeps the run inside the Phase 4 budget
# (~1.5 h on A100 per arm). Bump to None for a full pass when budget allows;
# the directionality of the delta is stable from N=50 already (Phase 2's
# 50-item holdout produced delta consistent with later 1k-row runs).
N_PER_CAT = 50
print(f"n_per_cat={N_PER_CAT}, total items per arm ≈ {N_PER_CAT * 11}")

In [ ]:
# Evaluate the base Qwen 3.5 4B. ~80 min on A100 at N_PER_CAT=50 with
# the 512 token cap (down from ~2h at the 768 default). Phase 2 found 256
# truncated mid-reasoning; 768 was a conservative ceiling. 512 splits the
# difference -- most BFCL completions land at 200-400 tokens once the
# model commits to a call, and the rare longer ones produce missed parses
# rather than wrong answers (they still score the same as a hard miss).
from tooltuned_qwen.eval.bfcl_holdout import run_bfcl_full
base_results = run_bfcl_full(
    "Qwen/Qwen3.5-4B",
    model_label="Qwen/Qwen3.5-4B",
    n_per_cat=N_PER_CAT,
    max_new_tokens=512,
    out_path="results/bfcl_intree/base/results.json",
)
print("base overall:", base_results["overall"])
# Drop GPU allocations before loading the tuned arm. Without this the
# second `from_pretrained` can OOM on lower-VRAM runtimes (L4 has 24 GB).
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Evaluate the fine-tuned adapter (Phase 1.3 finding #5: pass the adapter
# repo to `from_pretrained` directly, never via `load_adapter`).
# max_new_tokens=512 matches the base arm so the two are comparable.
from tooltuned_qwen.eval.bfcl_holdout import run_bfcl_full
tuned_results = run_bfcl_full(
    "sukhrobnurali/tooltuned-qwen-3.5-4b",
    model_label="tooltuned-qwen-3.5-4b",
    n_per_cat=N_PER_CAT,
    max_new_tokens=512,
    out_path="results/bfcl_intree/tuned/results.json",
)
print("tuned overall:", tuned_results["overall"])
# Save tuned_results to disk IMMEDIATELY (S9 lesson: per-category data
# evaporates when Colab disconnects; only the disk file + the printed
# overall survived). Also mirror to the private HF smoke repo so a
# runtime crash before the local download still preserves diagnostics.
import json, pathlib
pathlib.Path("results/bfcl_intree/tuned/results_full.json").write_text(
    json.dumps(tuned_results, indent=2), encoding="utf-8"
)
try:
    from huggingface_hub import HfApi
    HfApi(token=os.environ["HF_TOKEN"]).upload_file(
        path_or_fileobj="results/bfcl_intree/tuned/results_full.json",
        path_in_repo="results/bfcl_intree/tuned/results_full.json",
        repo_id="sukhrobnurali/tooltuned-qwen-3.5-4b-smoke",
        repo_type="model",
    )
    print("saved to disk + mirrored to smoke repo")
except Exception as e:
    print(f"local save OK; smoke-repo backup failed ({e!r}) -- download from sidebar now")
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Build the comparison + chart, gate-check on +3pp, regenerate + push
# the model card to HF if the gate passes. Gate failure halts here --
# Branch B in the project plan: write 0006-eval-debugging.md instead
# of pushing a sub-gate artifact.
from tooltuned_qwen.eval.compare import build_comparison
from tooltuned_qwen.hub.model_card import generate_card
from huggingface_hub import HfApi

results = build_comparison(
    base_results=base_results,
    tuned_results=tuned_results,
    out_dir="results/bfcl_intree/comparison",
    base_model_name="Qwen/Qwen3.5-4B",
    title="BFCL V3 single-turn -- base vs. fine-tuned Qwen 3.5 4B (in-tree eval)",
)
print(
    f"delta: {results['delta']*100:+.2f}pp  "
    f"(base {results['overall_base']*100:.1f}% -> tuned {results['overall_tuned']*100:.1f}%)"
)

if results["delta"] < 0.03:
    print(
        "GATE FAILED: delta below 3pp threshold. "
        "Write docs/decisions/0006-eval-debugging.md before pushing."
    )
else:
    card_path = generate_card(
        bfcl_results=results,
        training_config_path="configs/default.yaml",
        out_path="MODEL_CARD.md",
    )
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.upload_file(
        path_or_fileobj=card_path,
        path_in_repo="README.md",
        repo_id="sukhrobnurali/tooltuned-qwen-3.5-4b",
        repo_type="model",
    )
    api.upload_file(
        path_or_fileobj="results/bfcl_intree/comparison/bfcl_comparison.png",
        path_in_repo="bfcl_comparison.png",
        repo_id="sukhrobnurali/tooltuned-qwen-3.5-4b",
        repo_type="model",
    )
    print("model card + chart pushed to HF")